# Conflict-Induced Food Crisis Prediction — Task 1: Data Collection and Fusion

## Overview
This notebook collects, cleans, and fuses three independent data sources into a single panel dataset.
Every row in the output represents one administrative region in one month.

| Source | What it provides |
|---|---|
| ACLED | Georeferenced conflict events across Africa (1997–present) |
| FEWS NET | IPC food insecurity phase classifications per region |
| CHIRPS | Monthly precipitation — 35 years of African rainfall |

**Key design decisions:**
- Single authoritative merge path → `panel_dataset_production.csv`
- FEWS is the left table (only label-bearing region-months kept)
- Uganda excluded (ACLED at admin2, FEWS at admin1 — 99% unmatched)
- 5 countries absent from FEWS source: Burundi, Eritrea, Madagascar, Malawi, Zimbabwe
- No future leakage: `crisis_90d` uses only forward IPC shifts, features use current/past data

---
## Step 1 — Install and Import

In [ ]:
!pip install requests geopandas shapely rapidfuzz -q

import os, json, time, re, warnings
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
from pathlib import Path
from rapidfuzz import process, fuzz

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

OUTPUT_DIR = Path('/content/crisis_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory : {OUTPUT_DIR}')
print(f'Pandas version   : {pd.__version__}')
print('All libraries loaded successfully.')

Output directory : /content/crisis_outputs
Pandas version   : 2.2.2
All libraries loaded successfully.


---
## Step 2 — Google Drive Mount and Restore Backup

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

source_dir = Path('/content/drive/MyDrive/crisis_outputs_backup')
dest_dir   = OUTPUT_DIR
dest_dir.mkdir(parents=True, exist_ok=True)

if source_dir.exists():
    if not any(dest_dir.iterdir()):
        print(f'Restoring backup from {source_dir} ...')
        for item in source_dir.iterdir():
            d = dest_dir / item.name
            shutil.copytree(str(item), str(d), dirs_exist_ok=True) if item.is_dir() else shutil.copy2(str(item), str(d))
        print('Restore complete.')
    else:
        print(f'Destination {dest_dir} already has files — skipping restore.')
else:
    print(f'No backup found at {source_dir} — will run all steps from scratch.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Destination /content/crisis_outputs already has files — skipping restore.


---
## Step 3 — Configuration

In [ ]:
# Credentials
EMAIL    = 'dule.abera_mif24@nusaputra.ac.id'
PASSWORD = '$$Dul411829'

# Date range
START_DATE = '2018-01-01'
END_DATE   = '2026-01-01'

# All 20 target countries
ALL_COUNTRIES = [
    'Ethiopia', 'Somalia', 'Sudan', 'South Sudan', 'Kenya',
    'Uganda', 'Nigeria', 'Niger', 'Mali', 'Burkina Faso',
    'Chad', 'Cameroon', 'Mozambique', 'Zimbabwe', 'Malawi',
    'Democratic Republic of Congo', 'Central African Republic',
    'Madagascar', 'Eritrea', 'Burundi'
]

# Countries confirmed absent from FEWS NET source — excluded from production panel
FEWS_ABSENT   = {'Burundi', 'Eritrea', 'Madagascar', 'Malawi', 'Zimbabwe'}

# Uganda excluded: ACLED admin2 vs FEWS admin1 — 99% spatial mismatch
PANEL_EXCLUDE = FEWS_ABSENT | {'Uganda'}

# Countries used in the production panel
PANEL_COUNTRIES = [c for c in ALL_COUNTRIES if c not in PANEL_EXCLUDE]

print(f'All target countries    : {len(ALL_COUNTRIES)}')
print(f'Excluded from panel     : {sorted(PANEL_EXCLUDE)}')
print(f'Panel countries         : {len(PANEL_COUNTRIES)}')
print(f'Panel country list      : {PANEL_COUNTRIES}')

All target countries    : 20
Excluded from panel     : ['Burundi', 'Eritrea', 'Madagascar', 'Malawi', 'Uganda', 'Zimbabwe']
Panel countries         : 14
Panel country list      : ['Ethiopia', 'Somalia', 'Sudan', 'South Sudan', 'Kenya', 'Nigeria', 'Niger', 'Mali', 'Burkina Faso', 'Chad', 'Cameroon', 'Mozambique', 'Democratic Republic of Congo', 'Central African Republic']


---
## Step 4 — ACLED Data Collection

In [ ]:
BASE_URL = 'https://acleddata.com/api/acled/read'

def get_token(email, password):
    r = requests.post(
        'https://acleddata.com/oauth/token',
        data={'username': email, 'password': password,
              'grant_type': 'password', 'client_id': 'acled'},
        timeout=20
    )
    if r.status_code != 200:
        raise Exception(f'Token error {r.status_code}: {r.text[:200]}')
    print('Token acquired.')
    return r.json()['access_token']

acled_raw_path = OUTPUT_DIR / 'acled_africa_raw.csv'

if acled_raw_path.exists():
    print(f'Loading existing ACLED raw file: {acled_raw_path}')
    df = pd.read_csv(acled_raw_path)
    df['event_date'] = pd.to_datetime(df['event_date'])
    df['fatalities'] = pd.to_numeric(df['fatalities'], errors='coerce').fillna(0)
else:
    token   = get_token(EMAIL, PASSWORD)
    headers = {'Authorization': f'Bearer {token}'}
    all_events = []

    for country in ALL_COUNTRIES:
        page = 1
        country_events = []
        while True:
            params = {
                'country':          country,
                'event_date':       f'{START_DATE}|{END_DATE}',
                'event_date_where': 'BETWEEN',
                'fields':           'event_date|event_type|country|admin1|fatalities',
                'limit':            5000,
                'page':             page,
            }
            try:
                r = requests.get(BASE_URL, headers=headers, params=params, timeout=60)
                if r.status_code != 200:
                    print(f'  {country} page {page} HTTP {r.status_code} — skipped')
                    break
                data = r.json().get('data', [])
                if not data:
                    break
                country_events.extend(data)
                if len(data) < 5000:
                    break
                page += 1
                time.sleep(1)
            except Exception as e:
                print(f'  {country} page {page} error: {e}')
                break
        print(f'  {country}... {len(country_events):,} events')
        all_events.extend(country_events)

    if not all_events:
        raise Exception('No ACLED data returned.')

    df = pd.DataFrame(all_events)
    df['event_date'] = pd.to_datetime(df['event_date'])
    df['fatalities'] = pd.to_numeric(df['fatalities'], errors='coerce').fillna(0)
    df.to_csv(acled_raw_path, index=False)

print(f'\nTotal rows    : {len(df):,}')
print(f'Countries     : {df["country"].nunique()}')
print(f'Date range    : {df["event_date"].min().date()} to {df["event_date"].max().date()}')
print(f'Fatalities    : {df["fatalities"].sum():,.0f}')
print(f'Saved: {acled_raw_path}')
df.head(3)

Loading existing ACLED raw file: /content/crisis_outputs/acled_africa_raw.csv

Total rows    : 207,682
Countries     : 20
Date range    : 2018-01-01 to 2025-04-01
Fatalities    : 355,826
Saved: /content/crisis_outputs/acled_africa_raw.csv


,event_date,event_type,country,admin1,fatalities
0,2021-01-05,Protests,Ethiopia,Oromia,0
1,2021-01-07,Protests,Ethiopia,Oromia,0
2,2021-01-05,Violence against civilians,Ethiopia,Oromia,1


---
## Step 5 — ACLED Monthly Aggregation

In [ ]:
acled_raw = pd.read_csv(OUTPUT_DIR / 'acled_africa_raw.csv')
acled_raw['event_date'] = pd.to_datetime(acled_raw['event_date'], errors='coerce')
acled_raw = acled_raw.dropna(subset=['event_date'])
acled_raw['year_month']    = acled_raw['event_date'].dt.to_period('M')
acled_raw['country_clean'] = acled_raw['country'].astype(str).str.strip()
acled_raw['admin1_clean']  = acled_raw['admin1'].astype(str).str.strip().str.title()
acled_raw['fatalities']    = pd.to_numeric(acled_raw['fatalities'], errors='coerce').fillna(0)

BATTLE_TYPES   = {'Battles', 'Explosions/Remote violence'}
CIVILIAN_TYPES = {'Violence against civilians'}

monthly = (acled_raw
    .groupby(['country_clean', 'admin1_clean', 'year_month'])
    .apply(lambda g: pd.Series({
        'fatalities_30d':    g['fatalities'].sum(),
        'events_30d':        len(g),
        'battle_events':     g['event_type'].isin(BATTLE_TYPES).sum(),
        'civilian_violence': g['event_type'].isin(CIVILIAN_TYPES).sum(),
    }))
    .reset_index()
    .rename(columns={'country_clean': 'country', 'admin1_clean': 'admin1'})
)

# Fill full monthly timeline per region (zero-fill missing months)
all_months = pd.period_range(
    start=monthly['year_month'].min(),
    end=monthly['year_month'].max(),
    freq='M'
)
full_data = []
for (country, admin1), grp in monthly.groupby(['country', 'admin1']):
    idx = pd.MultiIndex.from_product(
        [[country], [admin1], all_months],
        names=['country', 'admin1', 'year_month']
    )
    grp2 = grp.set_index(['country', 'admin1', 'year_month']).reindex(idx)
    full_data.append(grp2)

monthly = pd.concat(full_data).reset_index()
monthly[['fatalities_30d', 'events_30d', 'battle_events', 'civilian_violence']] = \
    monthly[['fatalities_30d', 'events_30d', 'battle_events', 'civilian_violence']].fillna(0)

# Conflict trend: +1 rising, 0 stable, -1 falling vs prior 3-month average
monthly = monthly.sort_values(['country', 'admin1', 'year_month']).copy()
prev_avg = monthly.groupby(['country', 'admin1'])['fatalities_30d'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
threshold = 5
monthly['conflict_trend'] = 0
monthly.loc[
    (monthly['fatalities_30d'] > prev_avg + threshold) |
    ((prev_avg == 0) & (monthly['fatalities_30d'] > 0)),
    'conflict_trend'
] = 1
monthly.loc[
    (monthly['conflict_trend'] == 0) &
    ((monthly['fatalities_30d'] < prev_avg - threshold) |
     ((monthly['fatalities_30d'] == 0) & (prev_avg > 0))),
    'conflict_trend'
] = -1
monthly['conflict_trend'] = monthly['conflict_trend'].fillna(0).astype(int)
monthly['year_month'] = monthly['year_month'].astype(str)
monthly.to_csv(OUTPUT_DIR / 'acled_monthly.csv', index=False)

print('ACLED monthly aggregated:')
print(f'  Rows       : {len(monthly):,}')
print(f'  Countries  : {monthly["country"].nunique()}')
print(f'  Regions    : {monthly["admin1"].nunique()}')
print(f'  Fatalities : {monthly["fatalities_30d"].sum():,.0f}')
print(f'  Saved to   : {OUTPUT_DIR}/acled_monthly.csv')
monthly.head(3)

ACLED monthly aggregated:
  Rows       : 40,392
  Countries  : 20
  Regions    : 451
  Fatalities : 355,826
  Saved to   : /content/crisis_outputs/acled_monthly.csv


,country,admin1,year_month,fatalities_30d,events_30d,battle_events,civilian_violence,conflict_trend
0,Burkina Faso,Boucle Du Mouhoun,2018-01,0.0,1.0,1.0,0.0,0
1,Burkina Faso,Boucle Du Mouhoun,2018-02,0.0,0.0,0.0,0.0,0
2,Burkina Faso,Boucle Du Mouhoun,2018-03,1.0,1.0,1.0,0.0,1


---
## Step 6 — CHIRPS Rainfall Processing

In [ ]:
chirps_path = OUTPUT_DIR / 'chirps_monthly.csv'
chirps_raw  = pd.read_csv(chirps_path)

print(f'Loaded CHIRPS: {len(chirps_raw):,} rows')
print(f'Columns: {chirps_raw.columns.tolist()}')

# Build year_month if not present
if 'year_month' not in chirps_raw.columns:
    chirps_raw['year_month'] = (
        chirps_raw['year'].astype(str) + '-' +
        chirps_raw['month'].astype(str).str.zfill(2)
    )

chirps_raw = chirps_raw.rename(columns={'average_rainfall_mm': 'rainfall_mm'})

# Normalize case (lowercase for consistent merge key)
chirps_raw['country'] = chirps_raw['country'].astype(str).str.strip().str.lower()
chirps_raw['admin1']  = chirps_raw['admin1'].astype(str).str.strip().str.lower()
chirps_raw['year_month'] = chirps_raw['year_month'].astype(str)

# Critical fix: deduplicate before any computation
before = len(chirps_raw)
chirps_raw = chirps_raw.drop_duplicates(subset=['country', 'admin1', 'year_month'])
print(f'After dedup: {len(chirps_raw):,} rows (removed {before - len(chirps_raw):,} duplicates)')

# Drop stale anomaly columns — recompute cleanly
chirps_raw = chirps_raw.drop(columns=['rainfall_anomaly', 'is_drought', 'is_flood'], errors='ignore')

# Seasonal baseline per (country, admin1, calendar month)
chirps_raw['month_num'] = pd.to_numeric(chirps_raw['year_month'].str[-2:], errors='coerce')
chirps_raw = chirps_raw.dropna(subset=['month_num'])

seas = (
    chirps_raw
    .groupby(['country', 'admin1', 'month_num'])['rainfall_mm']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'seas_mean', 'std': 'seas_std'})
)
chirps_raw = chirps_raw.merge(seas, on=['country', 'admin1', 'month_num'], how='left')

chirps_raw['rainfall_anomaly'] = (
    (chirps_raw['rainfall_mm'] - chirps_raw['seas_mean'])
    / chirps_raw['seas_std'].replace(0, np.nan)
).round(3)

chirps_raw['is_drought'] = (chirps_raw['rainfall_anomaly'] < -0.5).astype(int)
chirps_raw['is_flood']   = (chirps_raw['rainfall_anomaly'] >  1.5).astype(int)

chirps_df = chirps_raw[[
    'country', 'admin1', 'year_month',
    'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood'
]].fillna({'rainfall_anomaly': 0, 'is_drought': 0, 'is_flood': 0})

chirps_df.to_csv(OUTPUT_DIR / 'chirps_monthly.csv', index=False)

unique_check = chirps_df[['country', 'admin1', 'year_month']].drop_duplicates().shape[0]
assert unique_check == len(chirps_df), f'CHIRPS still has duplicate keys! {unique_check} != {len(chirps_df)}'

print(f'\nCHIRPS panel:')
print(f'  Rows          : {len(chirps_df):,}')
print(f'  Countries     : {chirps_df["country"].nunique()}')
print(f'  Admin1 regions: {chirps_df["admin1"].nunique()}')
print(f'  Drought ratio : {chirps_df["is_drought"].mean():.1%}')
print(f'  Unique keys   : {unique_check:,} (no duplicates)')
print(f'  Saved to      : {OUTPUT_DIR}/chirps_monthly.csv')
chirps_df.head(3)

Loaded CHIRPS: 44,523 rows
Columns: ['country', 'admin1', 'year_month', 'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood']
After dedup: 44,523 rows (removed 0 duplicates)

CHIRPS panel:
  Rows          : 44,523
  Countries     : 20
  Admin1 regions: 451
  Drought ratio : 28.9%
  Unique keys   : 44,523 (no duplicates)
  Saved to      : /content/crisis_outputs/chirps_monthly.csv


,country,admin1,year_month,rainfall_mm,rainfall_anomaly,is_drought,is_flood
0,burkina faso,boucle du mouhoun,2018-01,44.612423,-1.406,1,0
1,burkina faso,boucle du mouhoun,2018-02,58.063046,1.990,0,1
2,burkina faso,boucle du mouhoun,2018-03,63.212420,1.041,0,0


---
## Step 7 — FEWS NET IPC Data

In [ ]:
REAL_IPC_PATH = OUTPUT_DIR / 'fews_net_ipc_admin1.csv'

TARGET_COUNTRIES = [
    'Ethiopia', 'Somalia', 'Sudan', 'South Sudan', 'Kenya',
    'Uganda', 'Nigeria', 'Niger', 'Mali', 'Burkina Faso',
    'Chad', 'Cameroon', 'Mozambique', 'Zimbabwe', 'Malawi',
    'Democratic Republic Of Congo', 'Central African Republic',
    'Madagascar', 'Eritrea', 'Burundi'
]

# All known FEWS country name variants → standard title-cased name
COUNTRY_MAP = {
    'Congo, The Democratic Republic of the':  'Democratic Republic Of Congo',
    'Democratic Republic of the Congo':        'Democratic Republic Of Congo',
    'Congo DR':                                'Democratic Republic Of Congo',
    'DR Congo':                                'Democratic Republic Of Congo',
    'Drc':                                     'Democratic Republic Of Congo',
    'Central African Republic':                'Central African Republic',
    'Madagascar':                              'Madagascar',
    'Eritrea':                                 'Eritrea',
    'Burundi':                                 'Burundi',
}

# Known FEWS → ACLED admin1 spelling corrections
ADMIN1_MAP = {
    ('Somalia',    'Banadir'):           'Banaadir',
    ('Somalia',    'Banaadir'):          'Banaadir',
    ('Somalia',    'Lower Juba'):        'Lower Juba',
    ('Somalia',    'Middle Juba'):       'Middle Juba',
    ('Somalia',    'Lower Shabelle'):    'Lower Shabelle',
    ('Somalia',    'Middle Shabelle'):   'Middle Shabelle',
    ('Nigeria',    'Fct Abuja'):         'Federal Capital Territory',
    ('Nigeria',    'Abuja'):             'Federal Capital Territory',
    ('Sudan',      'Blue Nile'):         'Blue Nile',
    ('Sudan',      'Khartoum'):          'Khartoum',
    ('South Sudan','Central Equatoria'): 'Central Equatoria',
}

print('Loading REAL FEWS NET IPC data...')
raw = pd.read_csv(REAL_IPC_PATH, low_memory=False)
print(f'  Raw rows           : {len(raw):,}')

# Step 1: Current Situation rows only
cs = raw[raw['scenario_name'].str.strip() == 'Current Situation'].copy()
print(f'  Current Situation  : {len(cs):,}')

# Step 2: Normalize country names
cs['country'] = (
    cs['country']
    .str.strip().str.title()
    .replace({k.title(): v for k, v in COUNTRY_MAP.items()})
)

# Diagnostic: which target countries are present
fews_countries = set(cs['country'].unique())
present = sorted(c for c in TARGET_COUNTRIES if c in fews_countries)
missing = sorted(c for c in TARGET_COUNTRIES if c not in fews_countries)
print(f'\n  Countries found in FEWS ({len(present)}/20): {present}')
print(f'  Countries ABSENT   ({len(missing)}/20): {missing}')
print(f'  (Absent = not monitored in this FEWS NET release)')

cs = cs[cs['country'].isin(TARGET_COUNTRIES)].copy()

# Step 3: Extract admin1 from geographic_unit_full_name
# Format: "district, admin1, country" — admin1 is second-to-last
def extract_admin1(full_name):
    parts = [p.strip() for p in str(full_name).split(',')]
    if len(parts) >= 3:
        return parts[-2]
    elif len(parts) == 2:
        return parts[0]
    return str(full_name)

cs['admin1'] = cs['geographic_unit_full_name'].apply(extract_admin1)
cs['admin1'] = cs['admin1'].str.strip().str.title()

# Step 4: Drop national-level rows (admin1 == country)
before = len(cs)
cs = cs[cs['admin1'] != cs['country']].copy()
print(f'\n  Dropped {before - len(cs):,} national-level rows (admin1 == country)')

# Step 5: Apply known admin1 spelling corrections
cs['admin1'] = cs.apply(
    lambda row: ADMIN1_MAP.get((row['country'], row['admin1']), row['admin1']),
    axis=1
)

# Step 6: Build year_month
cs['projection_start'] = pd.to_datetime(cs['projection_start'], errors='coerce')
cs = cs.dropna(subset=['projection_start'])
cs['year_month'] = cs['projection_start'].dt.to_period('M').astype(str)
cs = cs[(cs['year_month'] >= '2018-01') & (cs['year_month'] <= '2024-01')]

# Step 7: Aggregate to admin1 (max IPC phase = most severe district in region)
agg = (
    cs.groupby(['country', 'admin1', 'year_month'])
      .agg(ipc_phase=('value', 'max'))
      .reset_index()
)

print(f'\n  Uganda admin1 regions (before timeline fill): {agg[agg["country"]=="Uganda"]["admin1"].nunique()}')

# Step 8: Fill full monthly timeline per region (forward-fill gaps)
all_months = pd.period_range('2018-01', '2024-01', freq='M').astype(str)
full_data  = []
for (country, admin1), grp in agg.groupby(['country', 'admin1']):
    idx = pd.MultiIndex.from_product(
        [[country], [admin1], all_months],
        names=['country', 'admin1', 'year_month']
    )
    grp2 = grp.set_index(['country', 'admin1', 'year_month']).reindex(idx)
    grp2['ipc_phase'] = grp2['ipc_phase'].ffill().bfill()
    full_data.append(grp2)

fews = pd.concat(full_data).reset_index()
fews['ipc_phase'] = fews['ipc_phase'].fillna(2).round().astype(int)

# Double-filter: drop country-as-admin1 rows re-created by reindex
before = len(fews)
fews   = fews[fews['admin1'] != fews['country']].copy().reset_index(drop=True)
print(f'  Dropped {before - len(fews)} country-as-admin1 rows after reindex')

# Step 9: Build crisis_90d target (IPC >= 3 in any of next 3 months)
fews = fews.sort_values(['country', 'admin1', 'year_month']).reset_index(drop=True)
grp  = fews.groupby(['country', 'admin1'])
fews['ipc_next1'] = grp['ipc_phase'].shift(-1)
fews['ipc_next2'] = grp['ipc_phase'].shift(-2)
fews['ipc_next3'] = grp['ipc_phase'].shift(-3)
fews['crisis_90d'] = (
    fews[['ipc_next1', 'ipc_next2', 'ipc_next3']].max(axis=1) >= 3
).astype(int)
fews = fews.dropna(subset=['ipc_next1', 'ipc_next2', 'ipc_next3'])
fews = fews.drop(columns=['ipc_next1', 'ipc_next2', 'ipc_next3'])

# Step 10: Population placeholder (random for Phase 3+ regions)
rng = np.random.default_rng(seed=42)
fews['population_in_crisis'] = np.where(
    fews['ipc_phase'] >= 3,
    rng.integers(10000, 300000, len(fews)),
    0
)

fews.to_csv(OUTPUT_DIR / 'fewsnet_ipc.csv', index=False)

print(f'\nFEWS panel (REAL IPC labels):')
print(f'  Rows        : {len(fews):,}')
print(f'  Countries   : {fews["country"].nunique()}  ({len(missing)} absent from source)')
print(f'  Regions     : {fews["admin1"].nunique()}')
print(f'  Date range  : {fews["year_month"].min()} to {fews["year_month"].max()}')
print(f'  Crisis rate : {fews["crisis_90d"].mean():.1%}')
print(f'\nRegions per country:')
print(fews.groupby('country')[['admin1']].nunique().rename(columns={'admin1': 'regions'}).sort_values('regions', ascending=False).to_string())
print(f'\nSaved to: {OUTPUT_DIR}/fewsnet_ipc.csv')
fews.head(3)

Loading REAL FEWS NET IPC data...
  Raw rows           : 643,790
  Current Situation  : 82,669

  Countries found in FEWS (15/20): ['Burkina Faso', 'Cameroon', 'Central African Republic', 'Chad', 'Democratic Republic Of Congo', 'Ethiopia', 'Kenya', 'Mali', 'Mozambique', 'Niger', 'Nigeria', 'Somalia', 'South Sudan', 'Sudan', 'Uganda']
  Countries ABSENT   (5/20): ['Burundi', 'Eritrea', 'Madagascar', 'Malawi', 'Zimbabwe']
  (Absent = not monitored in this FEWS NET release)

  Dropped 274 national-level rows (admin1 == country)

  Uganda admin1 regions (before timeline fill): 18
  Dropped 0 country-as-admin1 rows after reindex

FEWS panel (REAL IPC labels):
  Rows        : 24,360
  Countries   : 14  (5 absent from source)
  Regions     : 345
  Date range  : 2018-01 to 2023-10
  Crisis rate : 30.4%

Regions per country:
                              regions
country                              
Sudan                              78
Nigeria                            63
Kenya               

,country,admin1,year_month,ipc_phase,crisis_90d,population_in_crisis
0,Burkina Faso,Boucle Du Mouhoun,2018-01,1,0,0
1,Burkina Faso,Boucle Du Mouhoun,2018-02,1,0,0
2,Burkina Faso,Boucle Du Mouhoun,2018-03,1,0,0


---
## Step 8 — Panel Merge

In [ ]:
# Admin1 name standardization (for merge key)
def standardize_admin1(name):
    if pd.isna(name) or str(name).strip() == '':
        return ''
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    replacements = {
        'southern nations nationalities and peoples': 'snnpr',
        'southern nations nationalities peoples':     'snnpr',
        'southern nations':                           'snnpr',
        'centre nord':    'centre-nord',
        'centre est':     'centre-est',
        'centre ouest':   'centre-ouest',
        'hauts bassins':  'hauts-bassins',
        'extreme nord':   'extreme-nord',
        'far north':      'extreme-nord',
    }
    for old, new in replacements.items():
        if old in name:
            return new
    return name

# Load all three sources
acled_m  = pd.read_csv(OUTPUT_DIR / 'acled_monthly.csv')
fews_m   = pd.read_csv(OUTPUT_DIR / 'fewsnet_ipc.csv')
chirps_m = pd.read_csv(OUTPUT_DIR / 'chirps_monthly.csv')

print(f'Loaded ACLED   : {len(acled_m):,} rows')
print(f'Loaded FEWS    : {len(fews_m):,} rows')
print(f'Loaded CHIRPS  : {len(chirps_m):,} rows')

# Build standardized merge keys
for df in [acled_m, fews_m, chirps_m]:
    df['country_key']    = df['country'].astype(str).str.strip().str.lower()
    df['admin1_key']     = df['admin1'].apply(standardize_admin1)
    df['year_month']     = df['year_month'].astype(str)

# Fuzzy mapping: ACLED admin1 → nearest FEWS admin1 per country
print('\nBuilding fuzzy admin1 mapping...')
fews_admin1_dict = fews_m.groupby('country_key')['admin1_key'].unique().to_dict()
admin1_map = {}

for country in acled_m['country_key'].unique():
    acled_names = acled_m[acled_m['country_key'] == country]['admin1_key'].unique()
    fews_names  = fews_admin1_dict.get(country, [])
    if len(fews_names) == 0:
        continue
    for ac_name in acled_names:
        if ac_name in fews_names:
            admin1_map[(country, ac_name)] = ac_name
        else:
            match = process.extractOne(
                ac_name, fews_names,
                scorer=fuzz.token_sort_ratio,
                score_cutoff=70
            )
            admin1_map[(country, ac_name)] = match[0] if match else ac_name

print(f'Fuzzy mapping created for {len(admin1_map)} ACLED region combinations')

acled_m['admin1_merge']  = acled_m.apply(
    lambda r: admin1_map.get((r['country_key'], r['admin1_key']), r['admin1_key']), axis=1)
chirps_m['admin1_merge'] = chirps_m.apply(
    lambda r: admin1_map.get((r['country_key'], r['admin1_key']), r['admin1_key']), axis=1)

# Deduplicate before merge (prevents cartesian explosion)
acled_dedup  = acled_m.drop_duplicates(subset=['country_key', 'admin1_merge', 'year_month'])
chirps_dedup = chirps_m.drop_duplicates(subset=['country_key', 'admin1_merge', 'year_month'])
fews_merge   = fews_m.copy()
fews_merge['admin1_merge'] = fews_merge['admin1_key']

print(f'\nACLED  deduped : {len(acled_dedup):,}')
print(f'CHIRPS deduped : {len(chirps_dedup):,}')
print(f'FEWS   rows    : {len(fews_merge):,}')

# Merge: FEWS left (keeps only labeled region-months)
# Step 1: FEWS + ACLED
panel = fews_merge.merge(
    acled_dedup[[
        'country_key', 'admin1_merge', 'year_month',
        'fatalities_30d', 'events_30d', 'battle_events',
        'civilian_violence', 'conflict_trend'
    ]],
    on=['country_key', 'admin1_merge', 'year_month'],
    how='left'
)

# Step 2: + CHIRPS
panel = panel.merge(
    chirps_dedup[[
        'country_key', 'admin1_merge', 'year_month',
        'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood'
    ]],
    on=['country_key', 'admin1_merge', 'year_month'],
    how='left'
)

# Fill missing conflict data (no events = zero conflict)
conflict_cols = ['fatalities_30d', 'events_30d', 'battle_events', 'civilian_violence', 'conflict_trend']
panel[conflict_cols] = panel[conflict_cols].fillna(0)

# IPC lag features
panel = panel.sort_values(['country_key', 'admin1_merge', 'year_month']).reset_index(drop=True)
grp = panel.groupby(['country_key', 'admin1_merge'])
panel['ipc_lag1'] = grp['ipc_phase'].shift(1)
panel['ipc_lag2'] = grp['ipc_phase'].shift(2)

# Seasonal features
panel['year_month_dt']    = pd.to_datetime(panel['year_month'] + '-01')
panel['month_num']        = panel['year_month_dt'].dt.month
panel['is_lean_season']   = panel['month_num'].isin([6, 7, 8, 9]).astype(int)
panel['is_harvest_season'] = panel['month_num'].isin([10, 11, 12]).astype(int)
panel['year']             = panel['year_month_dt'].dt.year
panel = panel.drop(columns=['year_month_dt', 'month_num'])

# Clean final column set
panel['country'] = panel['country_key']
panel['admin1']  = panel['admin1_merge']

# Drop rows missing rainfall or ipc_lag1 (insufficient history)
panel = panel.dropna(subset=['rainfall_anomaly', 'ipc_lag1']).reset_index(drop=True)

print(f'\nFull panel (all countries):')
print(f'  Rows           : {len(panel):,}')
print(f'  Countries      : {panel["country"].nunique()}')
print(f'  Regions        : {panel["admin1"].nunique()}')
print(f'  Date range     : {panel["year_month"].min()} to {panel["year_month"].max()}')
print(f'  Crisis rate    : {panel["crisis_90d"].mean():.1%}')
print(f'  Missing CHIRPS : {panel["rainfall_anomaly"].isnull().sum()}')
print(f'  Missing conflict: {panel["fatalities_30d"].isnull().sum()}')
panel.head(3)

Loaded ACLED   : 40,392 rows
Loaded FEWS    : 24,360 rows
Loaded CHIRPS  : 44,523 rows

Building fuzzy admin1 mapping...
Fuzzy mapping created for 383 ACLED region combinations

ACLED  deduped : 40,392
CHIRPS deduped : 44,523
FEWS   rows    : 24,360

Full panel (all countries):
  Rows           : 14,697
  Countries      : 12
  Regions        : 211
  Date range     : 2018-02 to 2023-10
  Crisis rate    : 33.1%
  Missing CHIRPS : 0
  Missing conflict: 0


,country,admin1,year_month,ipc_phase,crisis_90d,population_in_crisis,country_key,admin1_key,admin1_merge,fatalities_30d,events_30d,battle_events,civilian_violence,conflict_trend,rainfall_mm,rainfall_anomaly,is_drought,is_flood,ipc_lag1,ipc_lag2,is_lean_season,is_harvest_season,year
0,burkina faso,boucle du mouhoun,2018-02,1,0,0,burkina faso,boucle du mouhoun,boucle du mouhoun,0.0,0.0,0.0,0.0,0.0,58.063046,1.990,0.0,1.0,1.0,NaN,0,0,2018
1,burkina faso,boucle du mouhoun,2018-03,1,0,0,burkina faso,boucle du mouhoun,boucle du mouhoun,1.0,1.0,1.0,0.0,1.0,63.212420,1.041,0.0,0.0,1.0,1.0,0,0,2018
2,burkina faso,boucle du mouhoun,2018-04,1,0,0,burkina faso,boucle du mouhoun,boucle du mouhoun,0.0,2.0,1.0,1.0,-1.0,59.952377,1.307,0.0,0.0,1.0,1.0,0,0,2018


---
## Step 9 — Production Panel (Reliable Countries Only)

In [ ]:
panel_exclude_lower = {c.lower() for c in PANEL_EXCLUDE}
panel_prod = panel[~panel['country'].isin(panel_exclude_lower)].copy()

# Drop all intermediate merge key columns
panel_prod = panel_prod.drop(
    columns=['country_key', 'admin1_key', 'admin1_merge'],
    errors='ignore'
)

# Fix conflict_trend dtype (becomes float after reindex zero-fill)
panel_prod['conflict_trend'] = panel_prod['conflict_trend'].fillna(0).astype(int)

# feature_cols must NOT include ipc_phase — it is in the base columns already
feature_cols = [
    'ipc_lag1', 'ipc_lag2',
    'fatalities_30d', 'events_30d', 'battle_events',
    'civilian_violence', 'conflict_trend',
    'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood',
    'is_lean_season', 'is_harvest_season', 'year'
]

# Canonical column order — no duplicates
final_cols = [
    'country', 'admin1', 'year_month',
    'ipc_phase', 'population_in_crisis', 'crisis_90d'
] + feature_cols

panel_prod = panel_prod[final_cols].reset_index(drop=True)

panel_prod.to_csv(OUTPUT_DIR / 'panel_dataset.csv', index=False)

print('=== PRODUCTION PANEL ===')
print(f'  Rows           : {len(panel_prod):,}')
print(f'  Countries      : {panel_prod["country"].nunique()}')
print(f'  Regions        : {panel_prod["admin1"].nunique()}')
print(f'  Date range     : {panel_prod["year_month"].min()} to {panel_prod["year_month"].max()}')
print(f'  Crisis rate    : {panel_prod["crisis_90d"].mean():.1%}')
print(f'  Class imbalance: {(1-panel_prod["crisis_90d"].mean())/panel_prod["crisis_90d"].mean():.1f}:1 (safe:crisis)')
print(f'  Columns        : {panel_prod.columns.tolist()}')
print(f'  Duplicate cols : {[c for c in panel_prod.columns if panel_prod.columns.tolist().count(c) > 1]}')
print()
print('Countries excluded from panel:')
for c in sorted(PANEL_EXCLUDE):
    reason = 'absent from FEWS source' if c in FEWS_ABSENT else 'admin2/admin1 spatial mismatch'
    print(f'  {c}: {reason}')
print(f'\nSaved to: {OUTPUT_DIR}/panel_dataset.csv')
panel_prod.head(3)

=== PRODUCTION PANEL ===
  Rows           : 14,697
  Countries      : 12
  Regions        : 211
  Date range     : 2018-02 to 2023-10
  Crisis rate    : 33.1%
  Class imbalance: 2.0:1 (safe:crisis)
  Columns        : ['country', 'admin1', 'year_month', 'ipc_phase', 'population_in_crisis', 'crisis_90d', 'ipc_lag1', 'ipc_lag2', 'fatalities_30d', 'events_30d', 'battle_events', 'civilian_violence', 'conflict_trend', 'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood', 'is_lean_season', 'is_harvest_season', 'year']
  Duplicate cols : []

Countries excluded from panel:
  Burundi: absent from FEWS source
  Eritrea: absent from FEWS source
  Madagascar: absent from FEWS source
  Malawi: absent from FEWS source
  Uganda: admin2/admin1 spatial mismatch
  Zimbabwe: absent from FEWS source

Saved to: /content/crisis_outputs/panel_dataset.csv


,country,admin1,year_month,ipc_phase,population_in_crisis,crisis_90d,ipc_lag1,ipc_lag2,fatalities_30d,events_30d,battle_events,civilian_violence,conflict_trend,rainfall_mm,rainfall_anomaly,is_drought,is_flood,is_lean_season,is_harvest_season,year
0,burkina faso,boucle du mouhoun,2018-02,1,0,0,1.0,NaN,0.0,0.0,0.0,0.0,0,58.063046,1.990,0.0,1.0,0,0,2018
1,burkina faso,boucle du mouhoun,2018-03,1,0,0,1.0,1.0,1.0,1.0,1.0,0.0,1,63.212420,1.041,0.0,0.0,0,0,2018
2,burkina faso,boucle du mouhoun,2018-04,1,0,0,1.0,1.0,0.0,2.0,1.0,1.0,-1,59.952377,1.307,0.0,0.0,0,0,2018


---
## Step 10 — Quality Report

In [ ]:
ipc_lag1_corr = panel_prod[['ipc_lag1', 'crisis_90d']].corr().iloc[0, 1]
conflict_corr = panel_prod[['conflict_trend', 'crisis_90d']].corr().iloc[0, 1]
drought_corr  = panel_prod[['is_drought', 'crisis_90d']].corr().iloc[0, 1]

quality_report = {
    'panel_rows':            int(len(panel_prod)),
    'countries':             int(panel_prod['country'].nunique()),
    'regions':               int(panel_prod['admin1'].nunique()),
    'time_periods':          int(panel_prod['year_month'].nunique()),
    'date_start':            str(panel_prod['year_month'].min()),
    'date_end':              str(panel_prod['year_month'].max()),
    'crisis_rate':           float(round(panel_prod['crisis_90d'].mean(), 4)),
    'class_imbalance_ratio': float(round(
        (1 - panel_prod['crisis_90d'].mean()) / panel_prod['crisis_90d'].mean(), 2
    )),
    'missing_chirps_pct':    float(round(panel_prod['rainfall_anomaly'].isnull().mean() * 100, 2)),
    'missing_conflict_pct':  float(round(panel_prod['fatalities_30d'].isnull().mean() * 100, 2)),
    'total_fatalities':      int(panel_prod['fatalities_30d'].sum()),
    'drought_months_pct':    float(round(panel_prod['is_drought'].mean() * 100, 2)),
    'ipc_lag1_correlation':  float(round(ipc_lag1_corr, 4)),
    'conflict_trend_corr':   float(round(conflict_corr, 4)),
    'drought_corr':          float(round(drought_corr, 4)),
    'feature_columns':       feature_cols,
    'target_column':         'crisis_90d',
    'scale_pos_weight':      float(round(
        (1 - panel_prod['crisis_90d'].mean()) / panel_prod['crisis_90d'].mean(), 2
    )),
    'excluded_countries':    sorted(list(PANEL_EXCLUDE)),
    'fews_absent_countries': sorted(list(FEWS_ABSENT)),
}

with open(OUTPUT_DIR / 'data_quality_report.json', 'w') as f:
    json.dump(quality_report, f, indent=2)

print('=== TASK 1 COMPLETE ===')
print()
print(f'Panel dataset     : {quality_report["panel_rows"]:,} region-month observations')
print(f'Countries covered : {quality_report["countries"]} (of 20 target)')
print(f'Regions           : {quality_report["regions"]} admin-level-1 regions')
print(f'Date range        : {quality_report["date_start"]} to {quality_report["date_end"]}')
print(f'Crisis rate       : {quality_report["crisis_rate"]*100:.1f}%')
print(f'Class imbalance   : {quality_report["class_imbalance_ratio"]:.1f}:1 (safe:crisis)')
print(f'scale_pos_weight  : {quality_report["scale_pos_weight"]:.1f}  (use in XGBoost Task 3)')
print()
print('Feature correlations with crisis_90d:')
print(f'  ipc_lag1       : {ipc_lag1_corr:.4f}')
print(f'  conflict_trend : {conflict_corr:.4f}')
print(f'  is_drought     : {drought_corr:.4f}')
print()
print('Files saved:')
for f in sorted(OUTPUT_DIR.glob('*.csv')) + sorted(OUTPUT_DIR.glob('*.json')):
    print(f'  {f.name:<45} {f.stat().st_size/1024:>8.1f} KB')
print()
print('Pass panel_dataset.csv and data_quality_report.json to Task 2.')

=== TASK 1 COMPLETE ===

Panel dataset     : 14,697 region-month observations
Countries covered : 12 (of 20 target)
Regions           : 211 admin-level-1 regions
Date range        : 2018-02 to 2023-10
Crisis rate       : 33.1%
Class imbalance   : 2.0:1 (safe:crisis)
scale_pos_weight  : 2.0  (use in XGBoost Task 3)

Feature correlations with crisis_90d:
  ipc_lag1       : 0.7796
  conflict_trend : 0.0171
  is_drought     : 0.0445

Files saved:
  X_test.csv                                      1136.4 KB
  X_train.csv                                     1926.4 KB
  X_val.csv                                        493.1 KB
  acled_africa_raw.csv                           10334.1 KB
  acled_monthly.csv                               1776.4 KB
  chirps_monthly.csv                              2016.7 KB
  features_engineered.csv                         3882.3 KB
  features_manifest.csv                              2.2 KB
  fews_net_ipc_admin1 (1).csv                   311504.7 KB
  fews_net_ip

---
## Step 11 — Backup to Google Drive

In [ ]:
import shutil

drive_dest = Path('/content/drive/MyDrive/crisis_outputs_backup')
drive_dest.mkdir(parents=True, exist_ok=True)

print(f'Copying {OUTPUT_DIR} → {drive_dest} ...')
try:
    shutil.copytree(OUTPUT_DIR, drive_dest, dirs_exist_ok=True)
    print('Backup successful.')
    print(f'Location: {drive_dest}')
except Exception as e:
    print(f'Backup error: {e}')

Copying /content/crisis_outputs → /content/drive/MyDrive/crisis_outputs_backup ...
Backup successful.
Location: /content/drive/MyDrive/crisis_outputs_backup
